In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [2]:
# Applicazione di Machine Learning e Valutazione Modelli.
# Creo un solo dataframe (df) che conterrà il contenuto di entrmambi i file csv

DATASET_DIRECTORY ="../Data/"

df1 = pd.read_csv(os.path.join(DATASET_DIRECTORY, "Merged01.csv"))
df2 = pd.read_csv(os.path.join(DATASET_DIRECTORY, "Merged02.csv"))

df = pd.concat([df1, df2], axis=0, ignore_index=True)

In [3]:
# Da X_colums ho tolto le colonne che dallo studio svolto nel file study.ipynb sono risultate rumore
# colonne tolte: rst_flag_number,Variance,LLC,ARP,fin_count, syn_count, rst_count, ack_count, Header_Length,psh_flag_number e Protocol Type
X_columns = [
        'Time_To_Live', 'Rate',
       'fin_flag_number', 'syn_flag_number',
        'ack_flag_number', 'ece_flag_number',
       'cwr_flag_number', 'HTTP', 'HTTPS', 'DNS', 'Telnet', 'SMTP',
       'SSH', 'IRC', 'TCP', 'UDP','DHCP', 'ICMP', 'IGMP', 'IPv', 
       'Tot sum', 'Min', 'Max','AVG', 'Std', 'Tot size', 'IAT', 'Number',
]

# Effettuo un encoding della colonna target in modo da trasformare ogni tipo di attacco in un valore numerico
#faccio una mappatura a mano per essere più preciso
mappatura_label = {
    'BENIGN': 0,
    'DDOS-PSHACK_FLOOD': 1,
    'MIRAI-GREIP_FLOOD': 2,
    'DOS-UDP_FLOOD': 3,
    'DNS_SPOOFING': 4,
    'DDOS-ICMP_FLOOD': 5,
    'DDOS-TCP_FLOOD': 6,
    'DDOS-SYN_FLOOD': 7,
    'DDOS-UDP_FLOOD': 8,
    'MITM-ARPSPOOFING': 9,
    'DDOS-SYNONYMOUSIP_FLOOD': 10,
    'DOS-TCP_FLOOD': 11,
    'VULNERABILITYSCAN': 12,
    'DOS-SYN_FLOOD': 13,
    'DDOS-RSTFINFLOOD': 14,
    'DDOS-SLOWLORIS': 15,
    'DDOS-ICMP_FRAGMENTATION': 16,
    'MIRAI-GREETH_FLOOD': 17,
    'RECON-HOSTDISCOVERY': 18,
    'MIRAI-UDPPLAIN': 19,
    'RECON-PORTSCAN': 20,
    'DDOS-ACK_FRAGMENTATION': 21,
    'DDOS-UDP_FRAGMENTATION': 22,
    'RECON-OSSCAN': 23,
    'BACKDOOR_MALWARE': 24,
    'DOS-HTTP_FLOOD': 25,
    'XSS': 26,
    'DDOS-HTTP_FLOOD': 27,
    'BROWSERHIJACKING': 28,
    'SQLINJECTION': 29,
    'DICTIONARYBRUTEFORCE': 30,
    'COMMANDINJECTION': 31,
    'RECON-PINGSWEEP': 32,
    'UPLOADING_ATTACK': 33
}
df['Label'] = df['Label'].map(mappatura_label)
y_column = 'Label'

# rimuovo in oltre le righe del dataset che contengono valori inf o nulli nelle colonne : STD e Rete
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

X = df[X_columns]
y = df[y_column]

#import seaborn as sns
#corr_matrix_DF1 = df[X_columns].corr()
#plt.figure(figsize=(24, 18))
#sns.heatmap(corr_matrix_DF1, annot=True, cmap='coolwarm', linewidths=0.5,vmin=-1, vmax=1)


In [4]:
# Linear Regression

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error

#faccio un primo test con un 70%-30%, poi provo altre percentuali, sarà tutto salvato in delle note 
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("Valori con regressione lineare:'")
print(f"MSE: {mse*100:.2f}")
print(f"RMSE: {rmse*100:.2f}")

# --- RISULTATI REGRESSIONE LINEARE ---
# Test con 70%-30%: MSE: 948.66 | RMSE: 308.00
# Test con 80%-20%: MSE: 946.83 | RMSE: 307.71
# Test con 60%-40%: MSE: 950.21 | RMSE: 308.25

# OSSERVAZIONE:
# Le prestazioni del modello risultano scadenti, penso che il motivo sia un errata scelta del modello di ML
# la Regressione Lineare è progettata per stimare variabili continue, il nostro target, al contrario, è una variabile nominale. 
# I valori numerici assegnati agli attacchi fungono da semplici etichette e non possiedono  alcun significato matematico o ordinale.
# Concludo dicendo che, l'approccio regressivo è inefficace. Nelle sezioni successive il problema verrà affrontato utilizzando modelli differenti.

Valori con regressione lineare:'
MSE: 948.66
RMSE: 308.00


In [5]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(
    n_estimators=100, 
    max_depth=20,                          # Aggiungo un max_depth così da poter non far crushare la ram
    class_weight='balanced',                # Necessario per bilanciare gli output a causa dello sbilanciamento del dataset
    random_state=42, 
    n_jobs=2
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_rf)
precision = precision_score(y_test, y_pred_rf, average='weighted')
recall = recall_score(y_test, y_pred_rf, average='weighted')
f1 = f1_score(y_test, y_pred_rf, average='weighted')                # Il fatto che questo valore sia simile all'accuratezza fa intendere che il modello non va in overfiitng ma indovina anche i valori più rari
print(f"Accuracy: {accuracy:.2f}")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

# TEST
# Prova con n_estimators = 100 e test/train di 70%30%, Risultati: Accuracy: 0.78, Precision: 0.78, Recall: 0.77, F1-Score: 0.78 
# Prova con n_estimators = 100, test/train di 70%30% e max_depth:20, Risultati: Accuracy: 0.76, Precision: 0.78, Recall: 0.76, F1-Score: 0.76 T_Esec: 1:24
# Prova con n_estimators = 100, test/train di 80%20% e max_depth:20, Risultati: Accuracy: 0.76, Precision: 0.78, Recall: 0.76, F1-Score: 0.76 T_Esec: 1:47
# Prova con n_estimators = 200, test/train di 70%30% e max_depth:20, Risultati: Accuracy: 0.76, Precision: 0.78, Recall: 0.76, F1-Score: 0.76 T_Esec: 2:52
# Prova con n_estimators = 200, test/train di 80%20% e max_depth:20, Risultati: Accuracy: 0.76, Precision: 0.78, Recall: 0.76, F1-Score: 0.76 T_Esec: 3:10

# OSSERVAZIONE:
# Nel primo test senza limitazioni di profondità questo modello ha raggiunto le prestazioni massime (78%) ma purtroppo nei successivi test non mettere limitazioni
# ha portato al crash della RAM quindi per garantire stabilità ho dovuto aggiungerla peggiorando le prestazioni.
# La modifica del numero di alberi (aumentando il numero da 100 a 200) non ha portato alcun tipo di milgioria dal punto di vista della precisione ma ha solo allungato i tempi di esecuzione
# la mia conclusione finale è che il random forest è un modello di base che produce di buoni risultati ma i limiti di memoria legati alla creazione di 
# alberi paralleli e la difficoltà nell'estrarre pattern più complessi rende questo modello non non ancora sufficientemente accurato.


Accuracy: 0.76
Precision: 0.78
Recall: 0.76
F1-Score: 0.76


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [6]:
# XGboost
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_sample_weight

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Calcolo e assegno pesi differenti a seconda della rarità delle classi, questo sempre dovuto allo sbilanciamento del dataset
weight = compute_sample_weight(class_weight='balanced', y=y_train)
xgb_model = XGBClassifier(
    n_estimators=150,          
    max_depth=8,               # provo una profondità bassa suggerita da Xgboost
    learning_rate=0.2,         # Tasso di apprendimento 
    subsample=0.6,             # + Velocità : Usa solo il 60% delle righe per ogni albero
    colsample_bytree=0.6,      # + Velocità : Usa solo il 60% delle colonne per ogni albero
    tree_method='hist',        # Utilizzo un motodo ad istogramma per ridurre i tempi
    use_label_encoder=False,   # Dice a XGBoost che i target sono già numerici
    eval_metric='mlogloss',    # Indica quale formula matematica utilizzare per capire se sta migliorando o peggiorando durante la creazione degli alberi. essendo il target multiclasse uso mlogloss
    early_stopping_rounds=10,  # Se per X alberi di fila non migliora, si ferma.
    n_jobs=-1,                 # Usa tutti i core del processore
    random_state=42                   
)

xgb_model.fit(
    X_train, y_train, 
    sample_weight=weight,
    eval_set=[(X_test, y_test)],       # li uso per poter vedere le prestazioni
    verbose=10                         # stampa i progressi ogni 50 alberi
    )
y_pred_xgb = xgb_model.predict(X_test)

accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
precision_xgb = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
recall_xgb = recall_score(y_test, y_pred_xgb, average='weighted')
f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted')

print(f"Accuracy: {accuracy_xgb:.2f}")
print(f"Precision: {precision_xgb:.2f}")
print(f"Recall: {recall_xgb:.2f}")
print(f"F1-Score: {f1_xgb:.2f}")

# TEST
# Prova con n_estimators = 100 e test/train di 80%20% e learning_rate=0.1, Risultati: Accuracy: 0.74, Precision: 0.78, Recall: 0.74, F1-Score: 0.74 T_Esec: 3:37
# Prova con n_estimators = 100 e test/train di 80%20% e learning_rate=0.2, Risultati: Accuracy: 0.75, Precision: 0.78, Recall: 0.75, F1-Score: 0.75 T_Esec: 3:57
# Prova con n_estimators = 100 e test/train di 80%20% e learning_rate=0.5, Risultati: Accuracy: 0.74, Precision: 0.77, Recall: 0.74, F1-Score: 0.74 T_Esec: :49 -- In questo test è scattato l'early-stop a 24 giri
# Prova con n_estimators = 100 e test/train di 80%20% e learning_rate=0.5, Risultati: Accuracy: 0.74, Precision: 0.77, Recall: 0.74, F1-Score: 0.74 T_Esec: 1:29 -- in questo ho aumentato l'early-stop da 10 a 20

# Prova con n_estimators = 300 e test/train di 80%20% e learning_rate=0.2, Risultati: Accuracy: 0.76, Precision: 0.78, Recall: 0.76, F1-Score: 0.76 T_Esec 10:00 -- Noto che dal 100 giro in poi il miglioramento è minimo

# Prova con n_estimators = 150 e test/train di 80%20%, learning_rate=0.2 e max_depth=8, Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.76 -- Modifico il max_depth da 4 a 8, l'accuratezza è aumentata ma non notevolemnte
# Prova con n_estimators = 150 e test/train di 80%20%, learning_rate=0.2 e max_depth=16, Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.76 T_Esec: 3:20 --Modifico il max_depth da 8 a 16, nessun miglioramento

# Prova con n_estimators = 150 e test/train di 80%20%, learning_rate=0.2 e max_depth=8, Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.76 T_Esec: 6,16 -- Dopo aver rimosso subsample e colsample_bytree. nessun miglioramento, solo aumento di tempo di esecuzione

# OSSERVAZIONI
# Nonostante i vari test fatti il modello raggiunge uno stallo di accuratezza al 77%, L'uso di subsample=0.6 e colsample_bytree=0.6 ha ridotto notevolmente i tempi nonostante non abbia influito sull'efficacia del modello
# L'unico test rimasto da effettuare per verificare un eventuale aumento dell'efficacia di questi ultimi due modelli è aumentare il numero di alberi notevolemnte. ma questo causa un aumento importante dei tempi di esecuzione,
# rendendo obbbligatorio effettuare test su dataset di dimensioni notevolemnte minori (quindi con il rischio di perdere realisticità)
# Anche con questo modello i risultati non sono sufficienti endendo necessario il passaggio ad architetture di Deep Learning (Reti Neurali)


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xgboost/callback.py:385: UserWarning: [09:01:10] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  self.starting_round = model.num_boosted_rounds()


[0]	validation_0-mlogloss:1.87052
[10]	validation_0-mlogloss:0.73015
[20]	validation_0-mlogloss:0.54942
[30]	validation_0-mlogloss:0.49279
[40]	validation_0-mlogloss:0.47177
[50]	validation_0-mlogloss:0.46179
[60]	validation_0-mlogloss:0.45646
[70]	validation_0-mlogloss:0.45299
[80]	validation_0-mlogloss:0.45059
[90]	validation_0-mlogloss:0.44902
[100]	validation_0-mlogloss:0.44799
[110]	validation_0-mlogloss:0.44694
[120]	validation_0-mlogloss:0.44643
[130]	validation_0-mlogloss:0.44578
[140]	validation_0-mlogloss:0.44530
[149]	validation_0-mlogloss:0.44513
Accuracy: 0.77
Precision: 0.78
Recall: 0.77
F1-Score: 0.76


In [7]:
# Rete Neurale
from sklearn.preprocessing import StandardScaler       
import tensorflow as tf
from tensorflow import keras
from keras import optimizers, layers
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

X_train, X_temp, y_train, y_temp = train_test_split(X,y,test_size=0.2,random_state=42)
X_val, X_test, y_val, y_test = train_test_split (X_temp,y_temp, test_size=0.5, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)
num_classes = 34


model = keras.Sequential([
    layers.InputLayer(shape=(X_train.shape[1],)),

    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),                    # Spegne il 20% dei neuroni, previene l'overfitting
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"), 
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

#Applico un early stop anche in questo modello per evitare di sprecare tempo prezioso 
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', 
    patience=10, 
    restore_best_weights=True
)

#weight = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
#diz_weight = dict(enumerate(weight))            #Le reti neurali necessitano di un dizionario per i pesi

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=256,
    #class_weight=diz_weight,            
    verbose=1,
    callbacks=[early_stop]
)

y_pred_probs = model.predict(X_test)        # invece di restituire un valore unico la rete neurale restituisce la probabilità per ogni classe 
y_pred_rn = np.argmax(y_pred_probs, axis=1) # Prende la probabilità più alta

accuracy_rn = accuracy_score(y_test, y_pred_rn)
precision_rn = precision_score(y_test, y_pred_rn, average='weighted', zero_division=0)
recall_rn = recall_score(y_test, y_pred_rn, average='weighted')
f1_rn = f1_score(y_test, y_pred_rn, average='weighted')

print(f"Accuracy: {accuracy_rn:.2f}")
print(f"Precision: {precision_rn:.2f}")
print(f"Recall: {recall_rn:.2f}")
print(f"F1-Score: {f1_rn:.2f}")

#TEST
# Test con train/test/val di 70%15%15% e n_neuroni = 64 Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.74 T_esec: 4:46 
# Test con train/test/val di 80%10%10% e n_neuroni = 64 Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.74 T_esec: 2:43
# Test con train/test/val di 80%10%10% e n_neuroni = 256 Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77, F1-Score: 0.77 T_esec: 6:49 -- Aumento i numero da 64 a 256 e il dropuot da 0.2 a 0.3. Noto che l'accuratezza è rimasta uguale ma F1 è aumentato quindi le statistiche sono migliorate, a discapito dei tempi
# Test con train/test/val di 80%10%10% e n_neuroni = 256 batch_size 256 learning_rate 1e-3 Risultati: Accuracy: 0.77, Precision: 0.78, Recall: 0.77 F1-Score: 0.76 T_esec: 2:52  -- Rimuovo i pesi e alzo il learning rate testando solo con un aumento del batch_size. 

#OSSERVAZIONI
# Assegnare un peso alle classi per questo modello non si è rivelato un passagio utile.
# Anche con questo modello i risultati non superano il 77% di accuratezza ma si nota un aumnto per quanto riguarda F1_score e la netta riduzione dei tempi.


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


Epoch 1/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.7349 - loss: 0.5364 - val_accuracy: 0.7521 - val_loss: 0.4869
Epoch 2/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.7483 - loss: 0.4922 - val_accuracy: 0.7489 - val_loss: 0.4773
Epoch 3/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 18s 4ms/step - accuracy: 0.7541 - loss: 0.4832 - val_accuracy: 0.7626 - val_loss: 0.4615
Epoch 4/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.7639 - loss: 0.4676 - val_accuracy: 0.7691 - val_loss: 0.4538
Epoch 5/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 17s 4ms/step - accuracy: 0.7655 - loss: 0.4640 - val_accuracy: 0.7702 - val_loss: 0.4511
Epoch 6/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.7662 - loss: 0.4631 - val_accuracy: 0.7698 - val_loss: 0.4505
Epoch 7/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 20s 4ms/step - accuracy: 0.7652 - loss: 0.4633 - val_accuracy: 0.7702 - val_loss: 0.4534
Epoch 8/100
4566/4566 ━━━━━━━━━━━━━━━━━━━━ 16s 4ms/step - accuracy: 0.7653 -